# 02: Standalone Preprocessing, Entity Anonymization, Outlier Filtering & Stratified Splitting

This notebook executes the standalone preprocessing pipeline for CTI technique classification:

1. **Label Outlier Filtering**:
   - Filters out samples with 4 or more labels (keeping only samples with 1, 2, or 3 labels) to reduce noise from rare multi-label outliers while retaining over 99.7% of the dataset.
2. **Variable Entity Anonymization Protocol**:
   - Direct MITRE technique codes (`Txxxx`) were already neutralized in Notebook 01 to prevent label leakage.
   - Anonymizes 5 key variable technical entity groups: Vulnerabilities (`[CVE]`), IP addresses (`[IPV4]`), Web URLs (`[URL]`), File paths (`[FILE_PATH]`), and Cryptographic hashes (`[HASH]`).
   - Cleans HTML tags, unescapes entities, removes Markdown syntax, and normalizes Unicode while preserving natural casing and grammar for Transformer models (`Cleaned_Text`).
3. **Domain-Specific Tokenization**:
   - Extracts `Tokenized_Text` for TF-IDF baseline models using domain regex `[a-z0-9_\[\]]+(?:[./:-][a-z0-9_\[\]]+)*` to preserve technical tokens and special entity placeholders.
4. **Multi-Label Binarization & Stratified Train/Test Splitting**:
   - Fits `MultiLabelBinarizer` across the 378 parent technique target space.
   - Performs an 80% Train / 20% Test split using `MultilabelStratifiedShuffleSplit` (`random_state=42`) to preserve long-tail class distribution.
   - Exports processed datasets to `dataset/processed/` and preprocessing report to `results/02_preprocessing_report.json`.

In [1]:
import os
import sys
import json
import re
import html
import unicodedata
import pickle
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
import warnings
warnings.filterwarnings('ignore')

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

print('[INFO] Standalone preprocessing & stratification libraries loaded successfully.')

[INFO] Standalone preprocessing & stratification libraries loaded successfully.


In [2]:
# Define input and output directory paths
PROCESSED_DIR = Path('../dataset/processed')
RESULTS_DIR = Path('../results')

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

INPUT_MERGED_DATASET = PROCESSED_DIR / '01_merged_cti_dataset.csv'
OUTPUT_PROCESSED_DATASET = PROCESSED_DIR / '02_processed_cti_dataset.csv'
OUTPUT_TRAIN_CSV = PROCESSED_DIR / 'train.csv'
OUTPUT_TEST_CSV = PROCESSED_DIR / 'test.csv'
OUTPUT_BINARIZER_PKL = PROCESSED_DIR / 'multilabel_binarizer.pkl'
OUTPUT_REPORT_JSON = RESULTS_DIR / '02_preprocessing_report.json'

print(f"[INFO] Input Merged Dataset  : {INPUT_MERGED_DATASET.resolve()}")
print(f"[INFO] Output Processed Path : {PROCESSED_DIR.resolve()}")
print(f"[INFO] Preprocessing Report  : {OUTPUT_REPORT_JSON.resolve()}")

[INFO] Input Merged Dataset  : D:\Truong\FPT\SUMMER2026\AIC211\CTI_ATT&CK\cti_attck\dataset\processed\01_merged_cti_dataset.csv
[INFO] Output Processed Path : D:\Truong\FPT\SUMMER2026\AIC211\CTI_ATT&CK\cti_attck\dataset\processed
[INFO] Preprocessing Report  : D:\Truong\FPT\SUMMER2026\AIC211\CTI_ATT&CK\cti_attck\results\02_preprocessing_report.json


## 1. Load Dataset & Filter High Multi-Label Outliers (<= 3 Labels)

Filter out rare samples with 4 or more labels (keeping only samples with 1, 2, or 3 labels) to reduce noise in extreme multi-label training.

In [3]:
print("[STEP 1] Loading merged CTI dataset...")
df_raw = pd.read_csv(INPUT_MERGED_DATASET)
initial_sample_count = len(df_raw)
print(f"   [INFO] Initial dataset size: {initial_sample_count:,} samples.")

# Parse label arrays and calculate label counts per sample
label_lists_raw = [str(l).split(',') for l in df_raw['Labels']]
df_raw['Label_Count'] = [len(l) for l in label_lists_raw]

# Filter out samples with >= 4 labels
df_filtered = df_raw[df_raw['Label_Count'] <= 3].copy().reset_index(drop=True)
removed_outlier_count = initial_sample_count - len(df_filtered)

print(f"   [RESULT] Removed {removed_outlier_count} outlier samples with >= 4 labels.")
print(f"   [RESULT] Retained {len(df_filtered):,} valid samples with <= 3 labels ({len(df_filtered)/initial_sample_count*100:.2f}% of dataset).")

[STEP 1] Loading merged CTI dataset...
   [INFO] Initial dataset size: 22,384 samples.
   [RESULT] Removed 55 outlier samples with >= 4 labels.
   [RESULT] Retained 22,329 valid samples with <= 3 labels (99.75% of dataset).


## 2. Entity Anonymization Engine (Variable Entity Neutralization)

Neutralizes 5 variable technical entity groups (`[CVE]`, `[IPV4]`, `[URL]`, `[FILE_PATH]`, `[HASH]`), cleans HTML/Markdown formatting, and normalizes Unicode while preserving natural sentence grammar and casing for Transformer models.

In [4]:
# Define compiled regex patterns for variable technical entities
REG_CVE = re.compile(r'(?i)CVE-\d{4}-\d{4,7}')
REG_IPV4 = re.compile(r'\b(?:\d{1,3}\.){3}\d{1,3}\b')
REG_URL = re.compile(r'(?i)https?://[^\s]+|www\.[^\s]+')
REG_WIN_PATH = re.compile(r'[A-Za-z]:\\[^\s]+')
REG_UNIX_PATH = re.compile(r'/(?:[a-zA-Z0-9_\.-]+/)+[a-zA-Z0-9_\.-]+')
REG_HASH = re.compile(r'\b[a-fA-F0-9]{32}\b|\b[a-fA-F0-9]{40}\b|\b[a-fA-F0-9]{64}\b')
REG_HTML = re.compile(r'<[^>]+>')
REG_MARKDOWN = re.compile(r'[\*\_`#]')

def anonymize_cti_text_standalone(text):
    """
    Standalone entity anonymization pipeline:
    - HTML unescaping & NFKC Unicode normalization
    - Clean HTML tags and Markdown formatting
    - Anonymize CVE, IPV4, URL, FILE_PATH, and HASH entities
    - Preserve natural sentence structure and casing for Transformers
    """
    if pd.isna(text):
        return ""
    t = str(text)
    t = html.unescape(t)
    t = unicodedata.normalize('NFKC', t)
    t = REG_HTML.sub(' ', t)
    t = REG_MARKDOWN.sub(' ', t)
    
    # Substitute variable technical entities
    t = REG_CVE.sub(' [CVE] ', t)
    t = REG_URL.sub(' [URL] ', t)
    t = REG_WIN_PATH.sub(' [FILE_PATH] ', t)
    t = REG_UNIX_PATH.sub(' [FILE_PATH] ', t)
    t = REG_IPV4.sub(' [IPV4] ', t)
    t = REG_HASH.sub(' [HASH] ', t)
    
    # Clean step noise and whitespace
    t = re.sub(r'\b(step|phase)\s+\d+\b', ' ', t, flags=re.IGNORECASE)
    t = re.sub(r'\b(unknown|nan)\b', ' ', t, flags=re.IGNORECASE)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

# Test anonymization logic on sample
sample_test = "Attacker performed SQL Injection UNION SELECT 1,2,database() from 192.168.1.50 using payload at C:\\Windows\\System32\\cmd.exe exploiting CVE-2021-44228 and hash 5d41402abc4b2a76b9719d911017c592."
print("[TEST] Raw Sample      :", sample_test)
print("[TEST] Anonymized Output:", anonymize_cti_text_standalone(sample_test))

[TEST] Raw Sample      : Attacker performed SQL Injection UNION SELECT 1,2,database() from 192.168.1.50 using payload at C:\Windows\System32\cmd.exe exploiting CVE-2021-44228 and hash 5d41402abc4b2a76b9719d911017c592.
[TEST] Anonymized Output: Attacker performed SQL Injection UNION SELECT 1,2,database() from [IPV4] using payload at [FILE_PATH] exploiting [CVE] and hash [HASH] .


## 3. Domain-Specific CTI Tokenization

Extract lowercased token sequences (`Tokenized_Text`) preserving technical domain terms and special placeholders (`[ipv4]`, `[url]`, `[cve]`, `[file_path]`, `[hash]`) for TF-IDF baseline models.

In [5]:
CTI_TOKEN_PATTERN = r"[a-z0-9_\[\]]+(?:[./:-][a-z0-9_\[\]]+)*"

def cti_tokenizer(text):
    """Extract lowercased CTI technical tokens."""
    return re.findall(CTI_TOKEN_PATTERN, str(text).lower())

def tokenize_cti_text(text):
    """Convert text into space-delimited lowercased token string."""
    tokens = cti_tokenizer(text)
    return " ".join(tokens)

print("[STEP 2] Anonymizing entities and tokenizing text...")
df_filtered['Cleaned_Text'] = df_filtered['Cleaned_Text'].apply(anonymize_cti_text_standalone)
df_filtered['Tokenized_Text'] = df_filtered['Cleaned_Text'].apply(tokenize_cti_text)

# Remove any empty rows
df_processed = df_filtered[df_filtered['Cleaned_Text'].str.len() > 0].reset_index(drop=True)
print(f"   [SUCCESS] Preprocessing completed! Valid samples: {len(df_processed):,}")

[STEP 2] Anonymizing entities and tokenizing text...
   [SUCCESS] Preprocessing completed! Valid samples: 22,329


## 4. Binarize Target Label Matrix (378 Target Space)

Construct binary multi-label target matrix $Y \in \{0, 1\}^{N \times 378}$ and save `MultiLabelBinarizer` object.

In [6]:
label_lists = [str(l).split(',') for l in df_processed['Labels']]

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(label_lists)

print(f"[INFO] Target Matrix Y Shape : {Y.shape[0]:,} samples x {Y.shape[1]} labels")
print(f"[INFO] Active Unique Labels  : {len(mlb.classes_)} parent technique codes")

with open(OUTPUT_BINARIZER_PKL, 'wb') as f:
    pickle.dump(mlb, f)
print(f"[INFO] MultiLabelBinarizer instance exported to: {OUTPUT_BINARIZER_PKL}")

[INFO] Target Matrix Y Shape : 22,329 samples x 378 labels
[INFO] Active Unique Labels  : 378 parent technique codes
[INFO] MultiLabelBinarizer instance exported to: ..\dataset\processed\multilabel_binarizer.pkl


## 5. Multi-Label Stratified Train/Test Split (80% Train / 20% Test)

Perform iterative multi-label stratified splitting (`MultilabelStratifiedShuffleSplit`, `test_size=0.20`, `random_state=42`) to maintain balanced long-tail distribution across Train and Test sets.

In [7]:
print("[STEP 3] Performing Multi-Label Stratified Train/Test Split (80/20)...")
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_indices, test_indices = next(msss.split(df_processed['Cleaned_Text'].values, Y))

df_train = df_processed.iloc[train_indices].reset_index(drop=True)
df_test = df_processed.iloc[test_indices].reset_index(drop=True)

Y_train = Y[train_indices]
Y_test = Y[test_indices]

train_label_coverage = (Y_train.sum(axis=0) > 0).sum()
test_label_coverage = (Y_test.sum(axis=0) > 0).sum()

print(f"[SUCCESS] Stratified Split Summary:")
print(f"   - Train Set (80%)      : {len(df_train):,} samples ({len(df_train)/len(df_processed)*100:.2f}%)")
print(f"   - Test Set (20%)       : {len(df_test):,} samples ({len(df_test)/len(df_processed)*100:.2f}%)")
print(f"   - Train Label Coverage : {train_label_coverage} / {len(mlb.classes_)} labels ({train_label_coverage/len(mlb.classes_)*100:.2f}%)")
print(f"   - Test Label Coverage  : {test_label_coverage} / {len(mlb.classes_)} labels ({test_label_coverage/len(mlb.classes_)*100:.2f}%)")

[STEP 3] Performing Multi-Label Stratified Train/Test Split (80/20)...
[SUCCESS] Stratified Split Summary:
   - Train Set (80%)      : 17,876 samples (80.06%)
   - Test Set (20%)       : 4,453 samples (19.94%)
   - Train Label Coverage : 378 / 378 labels (100.00%)
   - Test Label Coverage  : 261 / 378 labels (69.05%)


## 6. Export Datasets & Preprocessing Report

Export finalized datasets (`02_processed_cti_dataset.csv`, `train.csv`, `test.csv`) to `dataset/processed/` and output JSON report to `results/02_preprocessing_report.json`.

In [8]:
df_processed.to_csv(OUTPUT_PROCESSED_DATASET, index=False, encoding='utf-8')
df_train.to_csv(OUTPUT_TRAIN_CSV, index=False, encoding='utf-8')
df_test.to_csv(OUTPUT_TEST_CSV, index=False, encoding='utf-8')

report_data = {
    "initial_samples": initial_sample_count,
    "removed_outliers_count": removed_outlier_count,
    "outlier_filter_rule": "Label_Count <= 3",
    "final_valid_samples": len(df_processed),
    "train_samples": len(df_train),
    "test_samples": len(df_test),
    "train_ratio": round(len(df_train) / len(df_processed), 4),
    "test_ratio": round(len(df_test) / len(df_processed), 4),
    "total_unique_target_labels": len(mlb.classes_),
    "train_label_coverage": int(train_label_coverage),
    "test_label_coverage": int(test_label_coverage),
    "random_seed": 42,
    "split_method": "MultilabelStratifiedShuffleSplit",
    "anonymized_entities": ["CVE", "IPV4", "URL", "FILE_PATH", "HASH"],
    "output_files": {
        "processed_dataset": "dataset/processed/02_processed_cti_dataset.csv",
        "train_set": "dataset/processed/train.csv",
        "test_set": "dataset/processed/test.csv",
        "multilabel_binarizer": "dataset/processed/multilabel_binarizer.pkl"
    }
}

with open(OUTPUT_REPORT_JSON, 'w', encoding='utf-8') as f:
    json.dump(report_data, f, ensure_ascii=False, indent=2)

print(f"[INFO] Exported processed dataset to : {OUTPUT_PROCESSED_DATASET}")
print(f"[INFO] Exported train set (80%) to   : {OUTPUT_TRAIN_CSV}")
print(f"[INFO] Exported test set (20%) to    : {OUTPUT_TEST_CSV}")
print(f"[INFO] Exported preprocessing report to: {OUTPUT_REPORT_JSON}")

[INFO] Exported processed dataset to : ..\dataset\processed\02_processed_cti_dataset.csv
[INFO] Exported train set (80%) to   : ..\dataset\processed\train.csv
[INFO] Exported test set (20%) to    : ..\dataset\processed\test.csv
[INFO] Exported preprocessing report to: ..\results\02_preprocessing_report.json


## 7. Sample Preview

Display first 5 rows from Train and Test datasets.

In [9]:
print("[PREVIEW] Train Set First 5 Rows:")
display(df_train[['Cleaned_Text', 'Tokenized_Text', 'Labels']].head(5))

print("\n[PREVIEW] Test Set First 5 Rows:")
display(df_test[['Cleaned_Text', 'Tokenized_Text', 'Labels']].head(5))

[PREVIEW] Train Set First 5 Rows:
                                        Cleaned_Text  ...       Labels
0  Authentication Bypass via SQL Injection Mobile...  ...  T1078,T1190
1  Union-Based SQL Injection AI Agents & LLM Expl...  ...        T1190
2  Error-Based SQL Injection AI Agents & LLM Expl...  ...        T1190
3  Blind SQL Injection AI Agents & LLM Exploits S...  ...        T1190
4  Second-Order SQL Injection AI Agents & LLM Exp...  ...        T1505

[5 rows x 3 columns]

[PREVIEW] Test Set First 5 Rows:
                                        Cleaned_Text  ...       Labels
0  Stored XSS (Persistent XSS) AI Agents & LLM Ex...  ...        T1059
1  Reflected XSS AI Agents & LLM Exploits Cross-S...  ...  T1059,T1189
2  DOM-Based XSS (Document Object Model XSS) AI A...  ...        T1059
3  Stored CSRF (via Stored XSS) AI Agents & LLM E...  ...  T1059,T1530
4  CSRF in JSON/REST API via Browser Auto-Request...  ...  T1059,T1530

[5 rows x 3 columns]
